# Exercícios de Mineração de Dados
Elisa Rachel Beninca Martins
## Parte 1: Z-Score (Análise Univariada)

### Exercício 1 — Sensor de temperatura ESP32

In [27]:
import numpy as np
from scipy.stats import zscore

temp = [45.5, 46.0, 45.2, 45.8, 46.1, 98.0, 45.9, 45.3]

# zscore() calcula, para cada valor, quantos desvios-padrão ele está afastado da média
# Fórmula: z = (x - média) / desvio_padrão
# Quanto mais distante da média, maior o z. Valores |z| > 2.5 são considerados anomalias
zscores = zscore(temp)

print("Temperatura |  Z-Score")
for t, z in zip(temp, zscores):
    print(f"  {t:5.1f}°C   |  {z:.4f}")

print("\nAnomalias detectadas (Z-Score > 2.5):")
for t, z in zip(temp, zscores):
    if z > 2.5:
        print(f"  Temperatura {t}°C (z = {z:.4f})")

Temperatura |  Z-Score
   45.5°C   |  -0.3886
   46.0°C   |  -0.3597
   45.2°C   |  -0.4060
   45.8°C   |  -0.3713
   46.1°C   |  -0.3540
   98.0°C   |  2.6453
   45.9°C   |  -0.3655
   45.3°C   |  -0.4002

Anomalias detectadas (Z-Score > 2.5):
  Temperatura 98.0°C (z = 2.6453)


### Exercício 2 — Monitoramento de voltagem de bateria

In [28]:
from scipy.stats import zscore

voltagem = [3.3, 3.2, 3.3, 3.4, 3.3, 1.2, 3.2, 3.3]

zscores = zscore(voltagem)

# Z-Score negativo significa que o valor está abaixo da média
print("    Voltagem    |  Z-Score   | Status")
for i, (v, z) in enumerate(zip(voltagem, zscores)):
    status = "Falha de energia" if z < -2.0 else "OK"
    print(f"  Disp {i+1}: {v}V  |  z={z:.4f}  |  {status}")

    Voltagem    |  Z-Score   | Status
  Disp 1: 3.3V  |  z=0.3972  |  OK
  Disp 2: 3.2V  |  z=0.2528  |  OK
  Disp 3: 3.3V  |  z=0.3972  |  OK
  Disp 4: 3.4V  |  z=0.5416  |  OK
  Disp 5: 3.3V  |  z=0.3972  |  OK
  Disp 6: 1.2V  |  z=-2.6359  |  Falha de energia
  Disp 7: 3.2V  |  z=0.2528  |  OK
  Disp 8: 3.3V  |  z=0.3972  |  OK


### Exercício 3 — Faturamento diário com erro de digitação

In [ ]:
import pandas as pd
from scipy.stats import zscore

vendas = [1200, 1350, 1250, 1300, 135000, 1280]

df = pd.DataFrame({'vendas': vendas})

df['z_score'] = zscore(df['vendas'])

print("DataFrame completo:")
print(df.to_string())

# np.abs() pega o valor absoluto do z_score para filtrar anomalias em ambos os lados
# Valores com |z| > 3 são considerados outliers
dados_limpos = df[df['z_score'].abs() <= 3]

print("\nDados limpos (sem outliers):")
print(dados_limpos.to_string())

# o outlier não foi removido porque, por ser um dataset pequeno, ele mesmo infla muito a média e o desvio, igual no exercício 4

DataFrame completo:
   vendas   z_score
0    1200 -0.448738
1    1350 -0.445729
2    1250 -0.447735
3    1300 -0.446732
4  135000  2.236067
5    1280 -0.447133

Dados limpos (sem outliers):
   vendas   z_score
0    1200 -0.448738
1    1350 -0.445729
2    1250 -0.447735
3    1300 -0.446732
4  135000  2.236067
5    1280 -0.447133


### Exercício 4 — Limitação do Z-Score com outlier colossal

In [30]:
import numpy as np
from scipy.stats import zscore

medidas = [10, 12, 11, 10, 10000]

zscores = zscore(medidas)
z_10000 = zscores[-1]  # z-score do último valor (10000)

media = np.mean(medidas)
desvio = np.std(medidas)

print(f"Média: {media:.2f}")
print(f"Desvio padrão: {desvio:.2f}")
print(f"Z-Score do valor 10000: {z_10000:.4f}")
print(f"Ultrapassa 3 sigmas? {'Sim' if abs(z_10000) > 3 else 'Não'}")

# O valor 10000 é tão grande que infla a média e o desvio padrão
# Com média 2009 e desvio 3996, o z do 10000 fica próximo de 2.0 — abaixo do limiar de 3
# Os valores normais (10, 11, 12) é que ficam com z negativo e perto de 0
# Isso demonstra que o Z-Score é sensível a outliers extremos em amostras pequenas
print("\nZ-Scores de todos os valores:")
for m, z in zip(medidas, zscores):
    print(f"  {m:6} -> z = {z:.4f}")

Média: 2008.60
Desvio padrão: 3995.70
Z-Score do valor 10000: 2.0000
Ultrapassa 3 sigmas? Não

Z-Scores de todos os valores:
      10 -> z = -0.5002
      12 -> z = -0.4997
      11 -> z = -0.4999
      10 -> z = -0.5002
   10000 -> z = 2.0000


## Parte 2: Isolation Forest (Análise Multivariada)

### Exercício 5 — Monitoramento de servidores (CPU vs RAM)

In [31]:
from sklearn.ensemble import IsolationForest

servidores = [[20, 30], [25, 35], [22, 32], [99, 95], [21, 31]]

# O modelo retorna:
#   1  -> ponto normal
#  -1  -> ponto anômalo (outlier)
modelo = IsolationForest(random_state=42)
modelo.fit(servidores)
predicoes = modelo.predict(servidores)

print("Servidor [CPU%, RAM%] -> Predição")
for servidor, pred in zip(servidores, predicoes):
    status = "ANOMALIA" if pred == -1 else "normal"
    print(f"  {servidor} -> {pred:2d}  ({status})")

Servidor [CPU%, RAM%] -> Predição
  [20, 30] ->  1  (normal)
  [25, 35] ->  1  (normal)
  [22, 32] ->  1  (normal)
  [99, 95] -> -1  (ANOMALIA)
  [21, 31] ->  1  (normal)


### Exercício 6 — Efeito do parâmetro `contamination`

In [32]:
import numpy as np
from sklearn.ensemble import IsolationForest

np.random.seed(42)

# 1000 pontos normais em torno de (0, 0)
normais = np.random.randn(1000, 2)

# 50 pontos absurdos com valores muito altos
absurdos = np.random.uniform(low=50, high=100, size=(50, 2))

dados = np.vstack([normais, absurdos])

# contamination=0.05 diz ao modelo: "espero que 5% dos dados sejam anomalias"
# contamination=0.20 diz: "espero que 20% sejam anomalias"
for contam in [0.05, 0.20]:
    modelo = IsolationForest(contamination=contam, random_state=42)
    preds = modelo.fit_predict(dados)
    n_anomalias = (preds == -1).sum()
    print(f"contamination={contam:.2f} -> {n_anomalias} anomalias detectadas")

print(f"\nTotal real de anomalias injetadas: 50 ({50/1050*100:.1f}% do dataset)")

contamination=0.05 -> 53 anomalias detectadas
contamination=0.20 -> 210 anomalias detectadas

Total real de anomalias injetadas: 50 (4.8% do dataset)


### Exercício 7 — Aluno com nota alta e muitas faltas

In [33]:
import numpy as np
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest

# [Nota, Faltas]
alunos = [[8, 2], [7, 4], [9, 1], [8, 3], [2, 25], [9, 25]]

notas  = [a[0] for a in alunos]
faltas = [a[1] for a in alunos]

z_notas = zscore(notas)

print("=== Z-Score individual de Notas ===")
for a, z in zip(alunos, z_notas):
    print(f"  {a} -> z_nota = {z:.4f}")

print("\n=== Isolation Forest (Nota + Faltas) ===")
modelo = IsolationForest(contamination=0.30, random_state=42)
preds = modelo.fit_predict(alunos)

for a, pred in zip(alunos, preds):
    status = "ANOMALIA" if pred == -1 else "normal"
    print(f"  Nota={a[0]}, Faltas={a[1]:2d} -> {status}")

# A floresta considera as duas dimensões juntas e detecta que [9, 25]
# está numa região de espaço onde ninguém mais está: nota alta + muitas faltas

=== Z-Score individual de Notas ===
  [8, 2] -> z_nota = 0.3459
  [7, 4] -> z_nota = -0.0692
  [9, 1] -> z_nota = 0.7609
  [8, 3] -> z_nota = 0.3459
  [2, 25] -> z_nota = -2.1443
  [9, 25] -> z_nota = 0.7609

=== Isolation Forest (Nota + Faltas) ===
  Nota=8, Faltas= 2 -> normal
  Nota=7, Faltas= 4 -> normal
  Nota=9, Faltas= 1 -> normal
  Nota=8, Faltas= 3 -> normal
  Nota=2, Faltas=25 -> ANOMALIA
  Nota=9, Faltas=25 -> ANOMALIA


### Exercício 8 — Base de alunos com DataFrame + Isolation Forest

In [34]:
import pandas as pd
from sklearn.ensemble import IsolationForest

df = pd.DataFrame({
    'idade':         [20, 22, 21, 23, 19, 150],
    'horas_estudo':  [ 8, 10,  9, 11,  7,   6],
    'nota_final':    [7.5, 8.0, 7.0, 8.5, 6.5, 5.0]
})

modelo = IsolationForest(contamination=0.15, random_state=42)

df['Outlier'] = modelo.fit_predict(df)

print("DataFrame completo:")
print(df.to_string())

print("\nLinha(s) detectada(s) como anomalia:")
print(df[df['Outlier'] == -1].to_string())

DataFrame completo:
   idade  horas_estudo  nota_final  Outlier
0     20             8         7.5        1
1     22            10         8.0        1
2     21             9         7.0        1
3     23            11         8.5        1
4     19             7         6.5        1
5    150             6         5.0       -1

Linha(s) detectada(s) como anomalia:
   idade  horas_estudo  nota_final  Outlier
5    150             6         5.0       -1


## Parte 3: Normalização

### Exercício 9 — Sensor de pressão e MinMaxScaler

In [35]:
from sklearn.preprocessing import MinMaxScaler

# Fórmula Min-Max: x_norm = (x - min) / (max - min)
# Para 200 psi com min=100 e max=500:
x, minimo, maximo = 200, 100, 500
manual = (x - minimo) / (maximo - minimo)
print(f"Cálculo manual: ({x} - {minimo}) / ({maximo} - {minimo}) = {manual:.4f}")

pressao = [[100], [200], [500]]
scaler = MinMaxScaler()
resultado = scaler.fit_transform(pressao)

print("\nResultado do MinMaxScaler:")
for psi, norm in zip(pressao, resultado):
    print(f"  {psi[0]} psi -> {norm[0]:.4f}")

print(f"\nO valor 200 psi normalizado: {resultado[1][0]:.4f} \nmanual: {manual:.4f}")

Cálculo manual: (200 - 100) / (500 - 100) = 0.2500

Resultado do MinMaxScaler:
  100 psi -> 0.0000
  200 psi -> 0.2500
  500 psi -> 1.0000

O valor 200 psi normalizado: 0.2500 
manual: 0.2500


### Exercício 10 — Escalonamento para comparação de clientes bancários

In [36]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame({
    'Cliente':  ['A', 'B'],
    'Saldo':    [1_000_000, 1_000_010],
    'Risco':    [0.1, 0.9]
})

print("Dados originais:")
print(df.to_string(index=False))

# Sem normalizar: a diferença de Saldo é 10 reais em escala de milhão (0.001%)
# O saldo domina completamente o cálculo, fazendo o risco parecer irrelevante
diff_saldo = abs(df['Saldo'][0] - df['Saldo'][1])
diff_risco = abs(df['Risco'][0] - df['Risco'][1])
print(f"\nSem normalização: diferença de saldo={diff_saldo}, diferença de risco={diff_risco}")
print("O saldo domina por um fator de", int(diff_saldo / diff_risco), "vezes")

scaler = MinMaxScaler()
normalizado = scaler.fit_transform(df[['Saldo', 'Risco']])
df_norm = pd.DataFrame(normalizado, columns=['Saldo_norm', 'Risco_norm'])
df_norm.insert(0, 'Cliente', ['A', 'B'])

print("\nApós normalização:")
print(df_norm.to_string(index=False))
print("\nAgora a diferença de saldo (10 reais) virou ~0.000001 e o risco domina com diferença de 0.8")

Dados originais:
Cliente   Saldo  Risco
      A 1000000    0.1
      B 1000010    0.9

Sem normalização: diferença de saldo=10, diferença de risco=0.8
O saldo domina por um fator de 12 vezes

Após normalização:
Cliente  Saldo_norm  Risco_norm
      A         0.0         0.0
      B         1.0         1.0

Agora a diferença de saldo (10 reais) virou ~0.000001 e o risco domina com diferença de 0.8


### Exercício 11 — Termômetro de câmara fria com valores negativos

In [37]:
from sklearn.preprocessing import MinMaxScaler

temps = [[-20], [-10], [0], [20]]

scaler = MinMaxScaler()
resultado = scaler.fit_transform(temps)

print("Temperatura original -> Normalizada")
for t, n in zip(temps, resultado):
    print(f"  {t[0]:4d}°C  ->  {n[0]:.4f}")

# O 0°C vira 0.5 após normalização
#   0.0 = o menor valor dos dados (-20°C)
#   1.0 = o maior valor dos dados (+20°C)
# O zero original (0°C) está no meio da escala (-20 a +20), então vira 0.5
# O novo 0.0 representa -20°C
print("\nO 0°C original virou:", resultado[2][0])

print("O novo 0.0 representa -20°C (o menor valor observado).")

Temperatura original -> Normalizada
   -20°C  ->  0.0000
   -10°C  ->  0.2500
     0°C  ->  0.5000
    20°C  ->  1.0000

O 0°C original virou: 0.5
O novo 0.0 representa -20°C (o menor valor observado).


### Exercício 12 — Pipeline completo: Z-Score + Normalização

In [38]:
import numpy as np
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler

producao = [100, 102, 98, 105, 500, 101]

# Passo 1: remover outliers com Z-Score
zscores = zscore(producao)
print("Z-Scores:")
for p, z in zip(producao, zscores):
    print(f"  {p:4d} -> z = {z:.4f}")

limiar = 2.0
dados_limpos = [p for p, z in zip(producao, zscores) if abs(z) <= limiar]
print(f"\nDados após remoção de outliers (|z| <= {limiar}): {dados_limpos}")

# Passo 2: normalizar os dados limpos
scaler = MinMaxScaler()
dados_array = np.array(dados_limpos).reshape(-1, 1)
normalizado = scaler.fit_transform(dados_array).flatten()

print("\nDados normalizados (0 a 1):")
for original, norm in zip(dados_limpos, normalizado):
    print(f"  {original} -> {norm:.4f}")

print("\n--- Demonstração do problema de normalizar COM outlier ---")
scaler2 = MinMaxScaler()
com_outlier = np.array(producao).reshape(-1, 1)
norm_com_outlier = scaler2.fit_transform(com_outlier).flatten()
print("Com o 500 incluso:")
for p, n in zip(producao, norm_com_outlier):
    print(f"  {p:4d} -> {n:.4f}  {'<-- outlier domina a escala' if p == 500 else ''}")

Z-Scores:
   100 -> z = -0.4552
   102 -> z = -0.4418
    98 -> z = -0.4687
   105 -> z = -0.4216
   500 -> z = 2.2358
   101 -> z = -0.4485

Dados após remoção de outliers (|z| <= 2.0): [100, 102, 98, 105, 101]

Dados normalizados (0 a 1):
  100 -> 0.2857
  102 -> 0.5714
  98 -> 0.0000
  105 -> 1.0000
  101 -> 0.4286

--- Demonstração do problema de normalizar COM outlier ---
Com o 500 incluso:
   100 -> 0.0050  
   102 -> 0.0100  
    98 -> 0.0000  
   105 -> 0.0174  
   500 -> 1.0000  <-- outlier domina a escala
   101 -> 0.0075  
